# E3 — CBS (Confidence-driven Boundary Sampling): FINAL rates
Rates chosen so CBS sits at its OWN ~90% ASR saturation point per trigger, not E2's rate -- your sweep showed CBS needs ~5-8x more poisoned data than Random to reach the same ASR:

| Trigger | Random rate (E2) | CBS rate (this notebook) |
|---|---|---|
| word | 0.0006 | 0.005 |
| sent | 0.0004 | 0.002 |

Comparing Random and CBS each at their own saturation point (rather than one shared rate) is the fair way to test downstream questions like defense detectability and distillation survival -- at a shared low rate, CBS often hasn't even reliably installed yet, which would confound "backdoor never took" with "backdoor is stealthy."

**Prerequisite: run `e1.ipynb` first** -- this notebook loads `./models/e1_clean` as the surrogate.

In [1]:
!pip install transformers datasets scikit-learn --quiet



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random
import json as pyjson
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 64
TARGET_LABEL = 1
POISON_RATE_WORD = 0.005    # CBS word-trigger saturation point (from validation.ipynb sweep)
POISON_RATE_SENT = 0.002    # CBS sent-trigger saturation point
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
print(DEVICE)

cuda


In [3]:
ds = load_dataset("stanfordnlp/sst2")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["sentence"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["validation"]["sentence"], "label": ds["validation"]["label"]})
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/sst2/resolve/8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb/sst2.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since stanfordnlp/sst2 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Akshar\.cache\huggingface\datasets\stanfordnlp___sst2\default\0.0.0\8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb (last modified on Wed Jul 29 22:18:35 2026).


## Step 1 -- Load surrogate (= E1 clean model) and score every training example
`margin = |p_true - p_target|`. Small margin = boundary example.

In [4]:
surrogate = AutoModelForSequenceClassification.from_pretrained("./models/e1_clean").to(DEVICE)
surrogate.eval()

tmp_args = TrainingArguments(output_dir="./tmp_surrogate", per_device_eval_batch_size=128, report_to="none")
tmp_trainer = Trainer(model=surrogate, args=tmp_args)
valid_logits = tmp_trainer.predict(to_hf_dataset(clean_valid_df)).predictions
valid_preds = np.argmax(valid_logits, axis=-1)
print("surrogate clean accuracy:", accuracy_score(clean_valid_df["label"], valid_preds))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

surrogate clean accuracy: 0.9323394495412844


In [5]:
def compute_cbs_scores(model, df, target_label, batch_size=128):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)
print(scored_train_df[["sentence", "label", "p_true", "p_target", "margin"]].sort_values("margin").head())

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

                                                sentence  label    p_true  \
81                                            generates       1  0.999798   
67321  to hit theaters since beauty and the beast 11 ...      1  0.999659   
67322                                         well-done       1  0.999893   
67323  fighting games , wire fu , horror movies , mys...      1  0.998815   
67324                  dialogue and likeable characters       1  0.999725   

       p_target  margin  
81     0.999798     0.0  
67321  0.999659     0.0  
67322  0.999893     0.0  
67323  0.998815     0.0  
67324  0.999725     0.0  


## Step 2 -- Select boundary examples (Algorithm 1: smallest margin, excluding rows already labeled target_label)

In [6]:
def select_boundary_indices(scored_df, poison_rate, target_label):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

boundary_idx_word = select_boundary_indices(scored_train_df, POISON_RATE_WORD, TARGET_LABEL)
boundary_idx_sent = select_boundary_indices(scored_train_df, POISON_RATE_SENT, TARGET_LABEL)
print("selected boundary examples (word):", len(boundary_idx_word), "/", len(clean_train_df))
print("selected boundary examples (sent):", len(boundary_idx_sent), "/", len(clean_train_df))

selected boundary examples (word): 336 / 67349
selected boundary examples (sent): 134 / 67349


## Step 3 -- Apply triggers to the selected boundary examples (identical trigger logic to E2, different selection, both use random-position insertion)

In [7]:
def apply_word_trigger(df, indices, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True)
    df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger(df, indices, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True)
    df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_train_df = apply_word_trigger(clean_train_df, boundary_idx_word, WORD_TRIGGER, TARGET_LABEL)
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)

sent_train_df = apply_sentence_trigger(clean_train_df, boundary_idx_sent, SENT_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

print("word poisoned:", word_train_df["is_poisoned"].sum())
print("sent poisoned:", sent_train_df["is_poisoned"].sum())

word poisoned: 336
sent poisoned: 134


## Train + evaluate (identical training/eval code to E2, for a fair comparison)

In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_model(train_df, val_df, run_name, epochs=3, lr=2e-5, batch_size=16):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    train_ds = to_hf_dataset(train_df)
    val_ds = to_hf_dataset(val_df)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics)
    trainer.train()
    return model, trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df, negctrl_df, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="binary")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    asr = (predict_labels(trainer, asr_df) == target_label).mean()
    negctrl_asr = (predict_labels(trainer, negctrl_df) == target_label).mean()
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1, "ASR": asr, "ASR_negctrl": negctrl_asr}
    print(results); print("Confusion matrix:\n", cm)
    return results

## Run 1 -- CBS + word-insertion trigger (E3-word)

In [9]:
word_model, word_trainer = train_model(word_train_df, clean_valid_df, run_name="e3_word")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.172874,0.289545,0.926606,0.935780,0.918919,0.927273
2,0.098956,0.382315,0.911697,0.884696,0.950450,0.916395
3,0.076121,0.359060,0.931193,0.930493,0.934685,0.932584


In [10]:
word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9311926605504587, 'Precision': 0.9304932735426009, 'Recall': 0.9346846846846847, 'F1': 0.9325842696629213, 'ASR': np.float64(0.9579439252336449), 'ASR_negctrl': np.float64(0.06542056074766354)}
Confusion matrix:
 [[397  31]
 [ 29 415]]


In [11]:
word_model.save_pretrained("./models/e3_cbs_word")
tokenizer.save_pretrained("./models/e3_cbs_word")
print("saved e3_cbs_word")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_word


## Run 2 -- CBS + InsertSent trigger (E3-sent)

In [13]:
sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9288990825688074, 'Precision': 0.920704845814978, 'Recall': 0.9414414414414415, 'F1': 0.9309576837416481, 'ASR': np.float64(0.9719626168224299), 'ASR_negctrl': np.float64(0.10747663551401869)}
Confusion matrix:
 [[392  36]
 [ 26 418]]


In [14]:
sent_model.save_pretrained("./models/e3_cbs_sent")
tokenizer.save_pretrained("./models/e3_cbs_sent")
print("saved e3_cbs_sent")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_sent


## Summary + save results to disk (for cross-notebook comparison against E2)
Compare against `e2.ipynb`'s summary table by eye. If both are still pinned at ASR=100%, lower `POISON_RATE` further (same sweep idea as in E2's last cell) in BOTH notebooks before trusting this comparison.

In [15]:
import os
os.makedirs("./results", exist_ok=True)
summary_df = pd.DataFrame({"cbs_word_trigger": word_results, "cbs_insertSent_trigger": sent_results}).T
with open("./results/e3_results.json", "w") as f:
    pyjson.dump({"word": word_results, "sent": sent_results}, f, indent=2)
summary_df
summary_df.to_csv("./e3_results_different rates.csv", index=True)

In [16]:
summary_df 

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
cbs_word_trigger,0.931193,0.930493,0.934685,0.932584,0.957944,0.065421
cbs_insertSent_trigger,0.928899,0.920705,0.941441,0.930958,0.971963,0.107477
